# Notebook 4 — Inference Demo

Interactive demo of the fine-tuned IAM Policy Generator.

Run this locally after training, or deploy to HuggingFace Spaces.

In [ ]:
import sys, json
sys.path.append('../src')
from inference import load_model, generate_policy

model, tokenizer = load_model('../iam-policy-adapter')
print('Model ready')

## Try it

In [ ]:
requirement = """
The DevOps team needs to be able to start and stop EC2 instances in the eu-west-1 region only.
They should not be able to terminate instances or modify security groups.
"""

result = generate_policy(requirement.strip(), model, tokenizer)
print(json.dumps(result, indent=2))

## Gradio UI (optional — deploy to HuggingFace Spaces)

In [ ]:
import gradio as gr

def ui_generate(requirement):
    result = generate_policy(requirement, model, tokenizer)
    return json.dumps(result, indent=2)

demo = gr.Interface(
    fn=ui_generate,
    inputs=gr.Textbox(lines=4, label='Access Control Requirement'),
    outputs=gr.Code(language='json', label='Generated IAM Policy + NIST Controls'),
    title='IAM Policy Generator',
    description='Fine-tuned Llama 3.2 3B — generates AWS IAM policies from plain English. **Review all outputs before production use.**',
    examples=[
        ['Finance team needs read-only access to S3 bucket prod-invoices. No delete. MFA required.'],
        ['DevOps can start and stop EC2 instances in eu-west-1 only. No terminate or security group changes.'],
        ['Data science team needs full access to SageMaker but no access to production S3 buckets.']
    ]
)

demo.launch()